<font size=8> Tiny CNN For Human Face Detector </font>

----

**Note:** We trained our CNN on Region of Interest (ROI) patches extracted using Haar Cascades. We used the LFW dataset for face mining and the PASS dataset for non-face mining. All detected ROIs were grouped, resized to 34x34, and enhanced via data augmentation. For instructions on generating your own train/val/test splits, please refer to the dataset_mining.ipynb document.

In [ ]:
# Install dependencies
!pip install opencv-python matplotlib tqdm numpy 

In [ ]:
# Imports
import cv2
import os
import glob
import random
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
from collections import defaultdict
from tqdm import tqdm

# Check versions
print(f"OpenCV version: {cv2.__version__}")
print(f"Matplotlib version: {matplotlib.__version__}")
print(f"NumPy version: {np.__version__}")
print("All packages installed successfully")

---

# Training set visualization:

**Faces:**

In [ ]:
NUM_SAMPLES = 12  # Number of face samples to show
DATASET_ROOT = "../dataset"
# Path to training faces
faces_dir = os.path.join(DATASET_ROOT, "train", "faces")

# Check if path exists
if not os.path.exists(faces_dir):
    print(f"ERROR: Training faces directory not found at: {faces_dir}")
    print("Please check your DATASET_ROOT path.")
else:
    # Get all face files
    face_files = [f for f in os.listdir(faces_dir) if f.endswith(".jpg")]

    if not face_files:
        print("No face images found in training set!")
    else:
        print(f"Found {len(face_files)} training face images")

        # Randomly select samples
        num_to_show = min(NUM_SAMPLES, len(face_files))
        selected = random.sample(face_files, num_to_show)

        # Create figure
        rows = (num_to_show + 3) // 4  # Calculate rows needed (4 columns)
        cols = min(4, num_to_show)

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))

        # Handle single subplot case
        if rows == 1 and cols == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = axes.reshape(1, -1)
        elif cols == 1:
            axes = axes.reshape(-1, 1)

        # Display each sample
        for i, f in enumerate(selected):
            row = i // cols
            col = i % cols

            img_path = os.path.join(faces_dir, f)
            img = cv2.imread(img_path)

            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[row, col].imshow(img_rgb)
                axes[row, col].set_title(f"Face {i+1}")
            else:
                axes[row, col].set_title(f"Error loading {f}")

            axes[row, col].axis("off")

        # Hide empty subplots
        for i in range(num_to_show, rows * cols):
            row = i // cols
            col = i % cols
            axes[row, col].axis("off")

        plt.suptitle(f"Training Face Samples (34×34)", fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

        # Print statistics
        print("\n Training Face Statistics:")
        print(f"Total training faces: {len(face_files)}")
        print(f"Image size: 34×34 pixels")
        print(f"Showing {num_to_show} random samples")

**Nonfaces:**

In [ ]:
NUM_SAMPLES = 12  # Number of non-face samples to show

# Path to training non-faces
nonfaces_dir = os.path.join(DATASET_ROOT, "train", "nonfaces")

# Check if path exists
if not os.path.exists(nonfaces_dir):
    print(f"ERROR: Training non-faces directory not found at: {nonfaces_dir}")
    print("Please check your DATASET_ROOT path.")
else:
    # Get all non-face files
    nonface_files = [f for f in os.listdir(nonfaces_dir) if f.endswith(".jpg")]

    if not nonface_files:
        print("No non-face images found in training set!")
    else:
        # Separate hard negatives (original) vs augmented
        hard_files = [f for f in nonface_files if "nonface_" in f and "aug" not in f]
        aug_files = [f for f in nonface_files if "aug" in f]

        # Randomly select samples to display
        num_to_show = min(NUM_SAMPLES, len(nonface_files))
        selected = random.sample(nonface_files, num_to_show)

        # Create figure
        rows = (num_to_show + 3) // 4  # Calculate rows needed (4 columns)
        cols = min(4, num_to_show)

        fig, axes = plt.subplots(rows, cols, figsize=(cols * 3, rows * 3))

        # Handle single subplot case
        if rows == 1 and cols == 1:
            axes = np.array([[axes]])
        elif rows == 1:
            axes = axes.reshape(1, -1)
        elif cols == 1:
            axes = axes.reshape(-1, 1)

        # Display each sample
        for i, f in enumerate(selected):
            row = i // cols
            col = i % cols

            img_path = os.path.join(nonfaces_dir, f)
            img = cv2.imread(img_path)

            if img is not None:
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                axes[row, col].imshow(img_rgb)
                axes[row, col].set_title(f"non-face {i+1}")
                    
            else:
                axes[row, col].set_title(f"Error")

            axes[row, col].axis("off")

        # Hide empty subplots
        for i in range(num_to_show, rows * cols):
            row = i // cols
            col = i % cols
            axes[row, col].axis("off")

        plt.suptitle(f"Training Non-Face Samples (34×34)", fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

        # Print statistics
        print("\nTraining Non-Face Statistics:")
        print(f"Total training non-faces: {len(nonface_files)}")
        print(f"Hard negatives (Haar false positives): {len(hard_files)}")
        print(f"Image size: 34×34 pixels")

# Dataset verification:

In [ ]:
def verify_dataset():
    print("DATASET VERIFICATION")

    total_faces = 0
    total_nonfaces = 0

    for split in ["train", "val", "test"]:
        faces_dir = os.path.join(DATASET_ROOT, split, "faces")
        nonfaces_dir = os.path.join(DATASET_ROOT, split, "nonfaces")

        f = len([f for f in os.listdir(faces_dir) if f.endswith(".jpg")])
        nf = len([f for f in os.listdir(nonfaces_dir) if f.endswith(".jpg")])

        total_faces += f
        total_nonfaces += nf

        print(f"\n{split.upper()}:")
        print(f"  Faces: {f}")
        print(f"  Non-faces: {nf}")
        print(f"  Total: {f + nf}")
        print(f"  Ratio (faces:nonfaces): 1:{nf/f:.2f}")

    print(f"TOTAL Faces: {total_faces}")
    print(f"TOTAL Non-faces: {total_nonfaces}")
    print(f"GRAND TOTAL: {total_faces + total_nonfaces}")


# Run verification
verify_dataset()

---

<font size="6"> CNN Training: </font>

**Input:** 34×34 grayscale patches from previous dataset

**Output:** Face (1) vs Non-face (0)

In [ ]:
%pip install torch torchvision scikit-learn

<font size="5"> **configurations** and **hyperparameters**: </font>

In [ ]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)

IMG_SIZE = 34
BATCH_SIZE = 32
EPOCHS = 20
LR = 1e-3
DEVICE = DEVICE = (
    "cuda" if torch.cuda.is_available() else "cpu"
)  # Our dataset is small, so CPU training should be valid

print(f"Configuration:")
print(f"  Dataset: {DATASET_ROOT}")
print(f"  Image size: {IMG_SIZE}×{IMG_SIZE}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Epochs: {EPOCHS}")
print(f"  Learning rate: {LR}")
print(f"  Device: {DEVICE}")

**FaceDataset** class:

In [ ]:
class FaceDataset(Dataset):
    def __init__(self, split):
        self.samples = []
        for label, cls in [(1, "faces"), (0, "nonfaces")]:
            path = os.path.join(DATASET_ROOT, split, cls)
            if not os.path.exists(path):
                print(f"Warning: {path} does not exist!")
                continue
            for f in os.listdir(path):
                if f.endswith(".jpg"):
                    self.samples.append((os.path.join(path, f), label))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
        img = img.astype(np.float32) / 255.0
        img = torch.from_numpy(img).unsqueeze(0)  # [1, 34, 34]
        return img, torch.tensor(label, dtype=torch.float32)

Dataset **loading**:

In [ ]:
train_ds = FaceDataset("train")
val_ds = FaceDataset("val")
test_ds = FaceDataset("test")

print(f"Dataset sizes:")
print(f"  Train: {len(train_ds)} images")
print(f"  Val: {len(val_ds)} images")
print(f"  Test: {len(test_ds)} images")

Class **balance:**

In [ ]:
train_faces = sum(1 for _, label in train_ds.samples if label == 1)
train_nonfaces = len(train_ds) - train_faces
print(f"\nTraining set balance:")
print(f"  Faces: {train_faces} ({train_faces/len(train_ds)*100:.1f}%)")
print(f"  Non-faces: {train_nonfaces} ({train_nonfaces/len(train_ds)*100:.1f}%)")

<font size="5"> TinyCNN archeticture: </font>

In [ ]:
class TinyCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 8, 3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(2)

        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(2)

        self.fc1 = nn.Linear(16 * 8 * 8, 32)
        self.relu3 = nn.ReLU()
        self.fc2 = nn.Linear(32, 1)

    def forward(self, x):
        x = self.relu1(self.conv1(x))
        x = self.pool1(x)
        x = self.relu2(self.conv2(x))
        x = self.pool2(x)
        x = x.view(x.size(0), -1)
        x = self.relu3(self.fc1(x))
        return torch.sigmoid(self.fc2(x)).squeeze(1)


# Initialize model
model = TinyCNN().to(DEVICE)
total_params = sum(p.numel() for p in model.parameters())
print(f"Model created with {total_params:,} parameters")

<font size="5"> Training and evaluation: </font>

In [ ]:
def train_epoch(model, loader, optimizer, loss_fn):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        out = model(x)
        loss = loss_fn(out, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)


@torch.no_grad()
def eval_model(model, loader):
    model.eval()
    ys, preds = [], []
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        preds.extend((out > 0.5).cpu().numpy())
        ys.extend(y.numpy())
    ys = np.array(ys)
    preds = np.array(preds)
    tn, fp, fn, tp = confusion_matrix(ys, preds).ravel()
    return {
        "acc": accuracy_score(ys, preds),
        "prec": precision_score(ys, preds),
        "rec": recall_score(ys, preds),
        "f1": f1_score(ys, preds),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

In [ ]:
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR)
loss_fn = nn.BCELoss()

# Tracking metrics for plotting
history = {"train_loss": [], "val_acc": [], "val_f1": [], "val_fp": [], "val_fn": []}

print("TRAINING STARTED")

best_f1 = 0
for epoch in range(EPOCHS):
    loss = train_epoch(model, train_loader, optimizer, loss_fn)
    val_metrics = eval_model(model, val_loader)

    # Store history
    history["train_loss"].append(loss)
    history["val_acc"].append(val_metrics["acc"])
    history["val_f1"].append(val_metrics["f1"])
    history["val_fp"].append(val_metrics["fp"])
    history["val_fn"].append(val_metrics["fn"])

    print(
        f"Epoch {epoch+1:2d}/{EPOCHS} | "
        f"Loss: {loss:.4f} | "
        f"Val Acc: {val_metrics['acc']:.3f} | "
        f"Val F1: {val_metrics['f1']:.3f} | "
        f"FP: {val_metrics['fp']:3d} | "
        f"FN: {val_metrics['fn']:3d}"
    )

    if val_metrics["f1"] > best_f1:
        best_f1 = val_metrics["f1"]
        torch.save(model.state_dict(), "tinycnn_face_best.pth")
        print(f"Saved best model (F1: {best_f1:.3f})")

<font size="5"> Visualization of training history: </font>

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Loss curve
axes[0, 0].plot(history["train_loss"], "b-", linewidth=2)
axes[0, 0].set_title("Training Loss", fontsize=14)
axes[0, 0].set_xlabel("Epoch")
axes[0, 0].set_ylabel("Loss")
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curve
axes[0, 1].plot(history["val_acc"], "g-", linewidth=2)
axes[0, 1].set_title("Validation Accuracy", fontsize=14)
axes[0, 1].set_xlabel("Epoch")
axes[0, 1].set_ylabel("Accuracy")
axes[0, 1].set_ylim([0, 1])
axes[0, 1].grid(True, alpha=0.3)

# F1 score curve
axes[1, 0].plot(history["val_f1"], "r-", linewidth=2)
axes[1, 0].set_title("Validation F1 Score", fontsize=14)
axes[1, 0].set_xlabel("Epoch")
axes[1, 0].set_ylabel("F1 Score")
axes[1, 0].set_ylim([0, 1])
axes[1, 0].grid(True, alpha=0.3)

# False positives/negatives
axes[1, 1].plot(history["val_fp"], "orange", linewidth=2, label="False Positives")
axes[1, 1].plot(history["val_fn"], "purple", linewidth=2, label="False Negatives")
axes[1, 1].set_title("Validation Errors", fontsize=14)
axes[1, 1].set_xlabel("Epoch")
axes[1, 1].set_ylabel("Count")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

plt.suptitle("Training Progress", fontsize=16)
plt.tight_layout()
plt.show()

---

# <font size="6"> Final Evaluation On Test Set Using The Saved Best Model:</font>

In [ ]:
# Load best model
model.load_state_dict(torch.load("tinycnn_face_best.pth", map_location=DEVICE))

# Evaluate on test set
test_metrics = eval_model(model, test_loader)

print("FINAL TEST RESULTS:")
print(f"Accuracy:  {test_metrics['acc']:.4f}")
print(f"Precision: {test_metrics['prec']:.4f}")
print(f"Recall:    {test_metrics['rec']:.4f}")
print(f"F1-Score:  {test_metrics['f1']:.4f}")
print("\nConfusion Matrix:")
print(f"           Predicted")
print(f"           Face  Non-face")
print(f"Actual-Face  {test_metrics['tp']:4d}   {test_metrics['fn']:4d}")
print(f"Non-face  {test_metrics['fp']:4d}      {test_metrics['tn']:4d}")

# Calculate false positive rate
fpr = test_metrics["fp"] / (test_metrics["fp"] + test_metrics["tn"])
fnr = test_metrics["fn"] / (test_metrics["fn"] + test_metrics["tp"])

print(f"\nFalse Positive Rate: {fpr*100:.2f}%")
print(f"False Negative Rate: {fnr*100:.2f}%")

ROC and AUC analysis for the best **threshold**:

In [ ]:
from sklearn.metrics import roc_curve, auc, roc_auc_score


@torch.no_grad()
def get_predictions_with_probs(model, loader):
    model.eval()
    all_probs = []
    all_labels = []

    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        all_probs.extend(out.cpu().numpy())
        all_labels.extend(y.numpy())

    return np.array(all_probs), np.array(all_labels)


# Get probabilities from test set
test_probs, test_labels = get_predictions_with_probs(model, test_loader)

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(test_labels, test_probs)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(10, 8))
plt.plot(fpr, tpr, color="darkorange", lw=2, label=f"ROC curve (AUC = {roc_auc:.3f})")
plt.plot([0, 1], [0, 1], color="navy", lw=2, linestyle="--", label="Random (AUC = 0.5)")

# Mark current threshold (0.5)
# Find closest threshold to 0.5
idx = np.argmin(np.abs(thresholds - 0.5))
plt.plot(fpr[idx], tpr[idx], "ro", markersize=8, label=f"Threshold = 0.5")

plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel("False Positive Rate", fontsize=12)
plt.ylabel("True Positive Rate (Recall)", fontsize=12)
plt.title("Receiver Operating Characteristic (ROC) Curve", fontsize=14)
plt.legend(loc="lower right")
plt.grid(True, alpha=0.3)
plt.show()

print(f"AUC Score: {roc_auc:.4f}")
print(f"\nAt threshold 0.5:")
print(f"  TPR (Recall): {tpr[idx]:.4f}")
print(f"  FPR: {fpr[idx]:.4f}")

# Show thresholds around operating point
print("\nThreshold analysis around 0.5:")
print(f"{'Threshold':>10} {'TPR':>8} {'FPR':>8} {'FP':>6} {'FN':>6}")
print("-" * 45)

for i in range(max(0, idx - 5), min(len(thresholds), idx + 6)):
    if i % 2 == 0:  # Show every other to avoid clutter
        fp = int(fpr[i] * len(test_labels[test_labels == 0]))
        fn = int((1 - tpr[i]) * len(test_labels[test_labels == 1]))
        print(f"{thresholds[i]:10.3f} {tpr[i]:8.3f} {fpr[i]:8.3f} {fp:6d} {fn:6d}")

---

Memory analysis of weights **(float32)**:

In [ ]:
def analyze_weight_sizes(model):
    print("\nMemory Analysis (float32)")
    print("-" * 50)

    total_float32_bytes = 0
    layer_sizes = []

    for name, param in model.named_parameters():
        num_elements = param.numel()
        bytes_float32 = num_elements * 4  # 4 bytes per float32
        total_float32_bytes += bytes_float32

        layer_sizes.append(
            {
                "name": name,
                "elements": num_elements,
                "float32_bytes": bytes_float32,
                "shape": tuple(param.shape),
            }
        )
        print(
            f"  {name:20}: {num_elements:6d} elements → {bytes_float32:7d} bytes {tuple(param.shape)}"
        )

    print("-" * 50)
    print(f"  TOTAL: {total_float32_bytes:,} bytes ({total_float32_bytes/1024:.2f} KB)")

    return total_float32_bytes, layer_sizes


float32_bytes, layer_info = analyze_weight_sizes(model)

---

<font size="6"> Quantization And Exportation Of Weights:  </font>

**Goal:** Quantize and export **float32** tensor to **int8**.

**Why:** float32 is memory heavy and slower for Embox (~4 bytes), while int8 (~1 byte).

In order to quantize from **float32** to **int8** we use the following formula:
$$
q = \text{round}(x \cdot \text{scale})
$$

where: 
$$
\text{scale} = \frac{127}{\text{max\_val}}
$$

In order to Dequantize (approximate) from **int8** to **float32** we use the following formula:
$$
x \approx \frac{q}{\text{scale}}
$$
 

In [ ]:
def quantize_to_int8(tensor):
    max_val = torch.max(torch.abs(tensor)).item()
    if max_val == 0:
        return torch.zeros_like(tensor, dtype=torch.int8).numpy(), 1.0
    scale = 127.0 / max_val
    quantized = torch.clamp(torch.round(tensor * scale), -128, 127).to(torch.int8)
    return quantized.numpy(), scale


def get_layer_info(layer):
    if isinstance(layer, nn.Conv2d):
        return {
            "type": "conv",
            "in_channels": layer.in_channels,
            "out_channels": layer.out_channels,
            "kernel_size": layer.kernel_size[0],
            "stride": layer.stride[0],
            "padding": layer.padding[0],
        }
    elif isinstance(layer, nn.Linear):
        return {
            "type": "fc",
            "in_features": layer.in_features,
            "out_features": layer.out_features,
        }
    return None


def export_quantized_weights(model):
    model.eval()
    os.makedirs("stm32_weights", exist_ok=True)

    layers = [
        ("conv1", model.conv1),
        ("conv2", model.conv2),
        ("fc1", model.fc1),
        ("fc2", model.fc2),
    ]

    total_float32_bytes = 0
    total_int8_bytes = 0
    layer_stats = []

    with open("stm32_weights/cnn_model.h", "w") as f:
        f.write("// TinyCNN for STM32 - int8 quantized\n")
        f.write("#ifndef CNN_MODEL_H\n#define CNN_MODEL_H\n\n")
        f.write("#include <stdint.h>\n\n")
        f.write(f"#define CNN_INPUT_SIZE {IMG_SIZE}\n")
        f.write("#define CNN_INPUT_CHANNELS 1\n")
        f.write("#define CNN_OUTPUT_SIZE 1\n\n")

        for name, layer in layers:
            info = get_layer_info(layer)
            if not info:
                continue

            # Quantize weights and bias
            weight_q, scale_w = quantize_to_int8(layer.weight.detach())
            bias_q, scale_b = quantize_to_int8(layer.bias.detach())

            # Calculate sizes
            float32_size = (layer.weight.numel() + layer.bias.numel()) * 4
            int8_size = weight_q.size + bias_q.size

            total_float32_bytes += float32_size
            total_int8_bytes += int8_size

            layer_stats.append(
                {
                    "name": name,
                    "float32_bytes": float32_size,
                    "int8_bytes": int8_size,
                    "savings": (1 - int8_size / float32_size) * 100,
                }
            )

            # Write weight file
            weight_file = f"stm32_weights/{name}_weight.h"
            with open(weight_file, "w") as wf:
                wf.write(f"// {name} weights (int8)\n")
                if info["type"] == "conv":
                    wf.write(
                        f"// Conv: {info['in_channels']}->{info['out_channels']}, "
                    )
                    wf.write(f"kernel {info['kernel_size']}x{info['kernel_size']}\n")
                else:
                    wf.write(f"// FC: {info['in_features']}->{info['out_features']}\n")
                wf.write(f"const int8_t {name}_weight[] = {{\n")
                flat = weight_q.flatten()
                for i, val in enumerate(flat):
                    wf.write(f"{int(val):4d}")
                    if i != len(flat) - 1:
                        wf.write(", ")
                    if (i + 1) % 16 == 0:
                        wf.write("\n")
                wf.write("};\n\n")
                wf.write(f"const float {name}_weight_scale = {1.0/scale_w:.6f}f;\n")

            # Write bias file
            bias_file = f"stm32_weights/{name}_bias.h"
            with open(bias_file, "w") as bf:
                bf.write(f"// {name} bias (int8)\n")
                bf.write(f"const int8_t {name}_bias[] = {{")
                bf.write(", ".join(f"{int(v):4d}" for v in bias_q))
                bf.write("};\n\n")
                bf.write(f"const float {name}_bias_scale = {1.0/scale_b:.6f}f;\n")

            f.write(f'#include "{name}_weight.h"\n')
            f.write(f'#include "{name}_bias.h"\n\n')

            if info["type"] == "conv":
                f.write(f"#define {name.upper()}_IN_CH {info['in_channels']}\n")
                f.write(f"#define {name.upper()}_OUT_CH {info['out_channels']}\n")
                f.write(f"#define {name.upper()}_KERNEL {info['kernel_size']}\n")
                f.write(f"#define {name.upper()}_STRIDE {info['stride']}\n")
                f.write(f"#define {name.upper()}_PADDING {info['padding']}\n\n")
            else:
                f.write(f"#define {name.upper()}_IN_FEATURES {info['in_features']}\n")
                f.write(
                    f"#define {name.upper()}_OUT_FEATURES {info['out_features']}\n\n"
                )

        f.write(f"#define CNN_TOTAL_SIZE {total_int8_bytes}\n")
        f.write("#endif // CNN_MODEL_H\n")

    print(f"\nExported weights to stm32_weights/")
    return layer_stats, total_float32_bytes, total_int8_bytes


# Run export
layer_stats, float32_total, int8_total = export_quantized_weights(model)

**Memory analysis** after quantization:

In [ ]:
print("\nQuantization Memory Savings")
print(f"{'Layer':<10} {'Float32 (bytes)':>15} {'Int8 (bytes)':>15} {'Savings':>10}")

for stat in layer_stats:
    print(
        f"{stat['name']:<10} {stat['float32_bytes']:>15,} {stat['int8_bytes']:>15,} {stat['savings']:>9.1f}%"
    )

print(
    f"{'TOTAL':<10} {float32_total:>15,} {int8_total:>15,} {(1-int8_total/float32_total)*100:>9.1f}%"
)

print(f"\nSTM32 Memory Footprint:")
print(f"  Model size: {int8_total:,} bytes ({int8_total/1024:.2f} KB)")

In [ ]:
# Save final model
torch.save(model.state_dict(), "tinycnn_face_final.pth")
print("Final model saved: tinycnn_face_final.pth")
print("STM32 weights saved in: stm32_weights/")
print("TRAINING COMPLETE")

---

 <font size="6"> Testing And Accuracy Comparison (int8 -> float32):</font>

In [ ]:
def load_quantized_weights():
    """Load the int8 weights from exported header files"""
    weights = {}

    # Helper to parse header file
    def parse_header(filename, var_name):
        with open(f"stm32_weights/{filename}", "r") as f:
            content = f.read()
            # Find the array
            start = content.find(f"const int8_t {var_name}[] = {{") + len(
                f"const int8_t {var_name}[] = {{"
            )
            end = content.find("};", start)
            values = content[start:end].strip().split(",")
            return np.array([int(v.strip()) for v in values if v.strip()])

    # Load all weights and biases
    weights["conv1_weight"] = parse_header("conv1_weight.h", "conv1_weight")
    weights["conv1_bias"] = parse_header("conv1_bias.h", "conv1_bias")
    weights["conv2_weight"] = parse_header("conv2_weight.h", "conv2_weight")
    weights["conv2_bias"] = parse_header("conv2_bias.h", "conv2_bias")
    weights["fc1_weight"] = parse_header("fc1_weight.h", "fc1_weight")
    weights["fc1_bias"] = parse_header("fc1_bias.h", "fc1_bias")
    weights["fc2_weight"] = parse_header("fc2_weight.h", "fc2_weight")
    weights["fc2_bias"] = parse_header("fc2_bias.h", "fc2_bias")

    # Load scales (from the same files)
    def get_scale(filename, var_name):
        with open(f"stm32_weights/{filename}", "r") as f:
            content = f.read()
            for line in content.split("\n"):
                if var_name in line:
                    return float(line.split("=")[1].strip().replace("f;", ""))
        return 1.0

    scales = {}
    scales["conv1_weight"] = get_scale("conv1_weight.h", "conv1_weight_scale")
    scales["conv1_bias"] = get_scale("conv1_bias.h", "conv1_bias_scale")
    scales["conv2_weight"] = get_scale("conv2_weight.h", "conv2_weight_scale")
    scales["conv2_bias"] = get_scale("conv2_bias.h", "conv2_bias_scale")
    scales["fc1_weight"] = get_scale("fc1_weight.h", "fc1_weight_scale")
    scales["fc1_bias"] = get_scale("fc1_bias.h", "fc1_bias_scale")
    scales["fc2_weight"] = get_scale("fc2_weight.h", "fc2_weight_scale")
    scales["fc2_bias"] = get_scale("fc2_bias.h", "fc2_bias_scale")

    return weights, scales


@torch.no_grad()
def test_quantized_accuracy(model, weights, scales, loader):
    """Run test set using quantized weights"""
    model.eval()
    ys, preds = [], []

    # Replace model weights with quantized versions
    with torch.no_grad():
        # Conv1
        model.conv1.weight.data = torch.tensor(
            weights["conv1_weight"].reshape(8, 1, 3, 3) * scales["conv1_weight"],
            dtype=torch.float32,
        )
        model.conv1.bias.data = torch.tensor(
            weights["conv1_bias"] * scales["conv1_bias"], dtype=torch.float32
        )

        # Conv2
        model.conv2.weight.data = torch.tensor(
            weights["conv2_weight"].reshape(16, 8, 3, 3) * scales["conv2_weight"],
            dtype=torch.float32,
        )
        model.conv2.bias.data = torch.tensor(
            weights["conv2_bias"] * scales["conv2_bias"], dtype=torch.float32
        )

        # FC1
        model.fc1.weight.data = torch.tensor(
            weights["fc1_weight"].reshape(32, 1024) * scales["fc1_weight"],
            dtype=torch.float32,
        )
        model.fc1.bias.data = torch.tensor(
            weights["fc1_bias"] * scales["fc1_bias"], dtype=torch.float32
        )

        # FC2
        model.fc2.weight.data = torch.tensor(
            weights["fc2_weight"].reshape(1, 32) * scales["fc2_weight"],
            dtype=torch.float32,
        )
        model.fc2.bias.data = torch.tensor(
            weights["fc2_bias"] * scales["fc2_bias"], dtype=torch.float32
        )

    # Run inference
    for x, y in loader:
        x = x.to(DEVICE)
        out = model(x)
        preds.extend((out > 0.5).cpu().numpy())
        ys.extend(y.numpy())

    ys = np.array(ys)
    preds = np.array(preds)
    tn, fp, fn, tp = confusion_matrix(ys, preds).ravel()

    return {
        "acc": accuracy_score(ys, preds),
        "prec": precision_score(ys, preds),
        "rec": recall_score(ys, preds),
        "f1": f1_score(ys, preds),
        "tp": int(tp),
        "fp": int(fp),
        "fn": int(fn),
        "tn": int(tn),
    }

In [ ]:
# Compare int8 with float

# Load quantized weights
weights, scales = load_quantized_weights()

# Test with quantized weights
quant_metrics = test_quantized_accuracy(model, weights, scales, test_loader)

print("\nFLOAT32 MODEL (original)")
print("-" * 50)
print(f"Accuracy:  {test_metrics['acc']:.4f}")
print(f"Precision: {test_metrics['prec']:.4f}")
print(f"Recall:    {test_metrics['rec']:.4f}")
print(f"F1-Score:  {test_metrics['f1']:.4f}")
print(f"FP: {test_metrics['fp']:3d} | FN: {test_metrics['fn']:3d}")

print("\nINT8 QUANTIZED MODEL (from exported headers)")
print("-" * 50)
print(f"Accuracy:  {quant_metrics['acc']:.4f}")
print(f"Precision: {quant_metrics['prec']:.4f}")
print(f"Recall:    {quant_metrics['rec']:.4f}")
print(f"F1-Score:  {quant_metrics['f1']:.4f}")
print(f"FP: {quant_metrics['fp']:3d} | FN: {quant_metrics['fn']:3d}")

print("\nQUANTIZATION IMPACT")
print("-" * 50)
print(f"Accuracy loss:  {test_metrics['acc'] - quant_metrics['acc']:.4f}")
print(f"F1 loss:        {test_metrics['f1'] - quant_metrics['f1']:.4f}")

---

## Model Architecture Explanation

**Input:** 34×34 grayscale image (resized Haar ROI)

**Layer 1: Conv2d (1→8, 3×3) + ReLU + MaxPool2d**
- Detects basic edges and textures
- 8 filters keep it lightweight
- MaxPool halves size to 17×17

**Layer 2: Conv2d (8→16, 3×3) + ReLU + MaxPool2d**
- Combines edges into facial features (eyes, nose, mouth)
- 16 filters capture more complex patterns
- MaxPool reduces to 8×8

**Layer 3: Fully Connected (1024→32) + ReLU**
- Flattens feature maps (8×8×16 = 1024)
- Compresses to 32 most important features

**Layer 4: Fully Connected (32→1) + Sigmoid**
- Final classification
- Output: probability (0-1) of being a human face

**Conclusion:**
- Progressive: edges → shapes → features → decision
- Tiny: ~34K parameters (34KB in int8)
- Fast: 2 conv layers keep computation low

---

### Alhajali Jalal

---